# 07. Paoay: exploratory aligned-model map

**Exploratory, not validated.** The main Paoay map (notebooks 01–05) applies the fixed January–April model. This notebook applies the **aligned matched** model instead. In that recipe, each pixel's four-month window starts at the month of its own detected **green-up** (the start of a sustained NDVI rise), as in training.

Why a separate notebook: the aligned recipe was compared with the fixed recipe on the external JECAM test, and it was chosen for this map only after those test scores were known. Choosing a model from test scores makes it a post-hoc choice, so this map is kept apart from the main Paoay outputs and carries no accuracy claim.

Inputs are local only: the three Paoay KMZ files, the Cloud Score+ half-month composites in `atok_imagery_root`, and the aligned matched package frozen by Training 02. Nothing is downloaded, fitted or changed in the main Paoay outputs. See the [workflow guide](../../docs/paoay_workflow.md).

## 1. Setup

Uses the same KMZ files, study-boundary name and context polygons as notebook 01. The model defaults to `aligned_matched` from the current Training 02 run, checked against that run's saved checksums. Set `paoay_aligned_model_manifest` in ignored `settings.local.json` to use a different package.

In [ ]:
# Load settings, locate the KMZ files, the Cloud Score+ composites and the aligned matched model
from pathlib import Path
import json
import sys

repository_root = next((folder for folder in [Path.cwd(), *Path.cwd().parents]
                        if (folder / "pyproject.toml").is_file() and (folder / "src/crop_pipeline").is_dir()), None)
if repository_root is None:
    raise FileNotFoundError("Open this notebook from inside the project repository")
if str(repository_root / "src") not in sys.path:
    sys.path.insert(0, str(repository_root / "src"))
from crop_pipeline.paoay import aligned
from crop_pipeline.notebook_display import show_table

settings_path = repository_root / "settings.local.json"
local_settings = json.loads(settings_path.read_text()) if settings_path.exists() else {}
def local_path(value):
    path = Path(value).expanduser()
    return path.resolve() if path.is_absolute() else (repository_root / path).resolve()

kmz_root = local_path(local_settings.get("paoay_kmz_root", "./data/paoay"))
kmz_paths = [kmz_root / name for name in ("00 - Region of Interest.kmz", "00 - Atok Boundary.kmz", "Crop Area.kmz")]
roi_name = local_settings.get("paoay_roi_name", "Region of Interest")
context_names = list(local_settings.get("paoay_context_names", []))
imagery_root = local_path(local_settings["atok_imagery_root"])
report_folder = local_path(local_settings.get("report_root", "./sen4agrinet_work/inventory"))
output_root = report_folder.parent / "paoay/aligned_exploratory"
explicit_model = local_settings.get("paoay_aligned_model_manifest")
model_manifest = local_path(explicit_model) if explicit_model else aligned.default_model(repository_root)
print("Model package:", model_manifest)
print("Composites:", imagery_root)
print("Outputs:", output_root)

## 2. Choose the year and build the map

The year is chosen by a rule fixed before mapping: all 24 half-month composites must exist, and among those years the one with the highest mean share of clear study pixels is used (ties go to the later year). Model predictions play no part in the choice.

The same steps as training then run for every pixel whose centre is inside the study boundary:

1. **Green-up detection** with the training detector and settings (relative threshold 0.2, minimum amplitude 0.15).
2. **Onset month:** the detected onset must fall within one calendar month; otherwise training required a human review, so the pixel gets no window here.
3. **Automatic timing checks** from training, unchanged: onset gap at most 20 days, sustained later growth, an NDRE rise of at least 0.02, and the same cycle and onset month at thresholds 0.15, 0.20 and 0.25.
4. **Window:** four calendar months from the onset month, inside the chosen year. When a pixel has several qualified cycles, the earliest one whose eight half-months are all clear is used (training's `earliest_complete` policy).
5. **Features and prediction:** the 72 saved-model inputs use the training formulas; predictions use the Paoay rule (top probability ≥ 0.60 and margin ≥ 0.10, otherwise Unknown crop type).

Differences from training are listed in the bundle's `deviations_from_training` and printed below. A bundle with identical inputs is reused, never overwritten.

In [ ]:
# Run the exploratory aligned map (or reuse the saved bundle with the same inputs) and load it
manifest_path = aligned.run(kmz_paths, roi_name, context_names, imagery_root, model_manifest, output_root)
bundle = aligned.load(manifest_path)
manifest = bundle["manifest"]
print("Bundle:", manifest_path)
print("Year chosen by the completeness rule:", manifest["identity"]["year"])
show_table(bundle["year_completeness"], title="Half-month completeness of the study area by year")
print("Differences from the training recipe:")
for item in manifest["deviations_from_training"]:
    print(" -", item)

## 3. Where pixels drop out

Every study pixel ends in one status: a crop, Unknown crop type, **No qualified green-up window** (no detected green-up passed the detector, onset-month and timing rules), or **Window not cloud-free** (a qualified window exists, but at least one of its eight half-months is missing in that pixel). Missing half-months are never filled in. The second table counts detector and timing outcomes per candidate cycle, so one pixel can appear more than once.

In [ ]:
# Count study pixels by final status and candidate cycles by detector/timing outcome
from collections import Counter
codes = bundle["codes"]
names = {code: name for name, code in manifest["codes"].items()}
status_counts = Counter(codes[codes > 0].tolist())
show_table([{"status": names[code], "pixels": status_counts.get(code, 0),
             "share_of_study_pixels": f"{status_counts.get(code, 0) / manifest['study_pixels']:.1%}"}
            for code in sorted(names)], title="Study pixels by final status")
show_table([{"outcome": outcome, "candidate_cycles": count}
            for outcome, count in sorted(manifest["window_outcomes"].items(), key=lambda item: -item[1])],
           title="Candidate cycles by detector and timing outcome")

## 4. Map and window start months

The map shows each pixel's final status with the reference-area outlines. The bar chart counts predicted pixels (crops and Unknown) by the month their window starts. Both PNGs are saved in the bundle's `previews/` folder.

In [ ]:
# Draw the exploratory map and the window-start chart, and save both as PNGs
from IPython.display import display
import matplotlib.pyplot as plt

roi, reference_areas, _ = aligned.study_area(kmz_paths, roi_name, context_names, manifest["grid"]["crs"])
previews = manifest_path.parent / "previews"
previews.mkdir(exist_ok=True)
for name, figure in (("aligned_map", aligned.plot_map(bundle, reference_areas)), ("window_start_months", aligned.plot_onsets(bundle))):
    figure.savefig(previews / f"{name}.png", dpi=150, bbox_inches="tight", facecolor="white")
    display(figure)
    plt.close(figure)
print("Previews:", previews)

## 5. Reference areas

Pixel counts by status for the whole study boundary and for each reference area (pixel centres inside the polygon). Reference-area notes are unverified calendars, so agreement or disagreement here is not an accuracy measure.

In [ ]:
# Show pixel status counts for the study boundary and each reference area
show_table(bundle["area_summary"], title="Pixels by status and area")

## 6. Limits

- The model choice used the external test scores, so this map cannot also serve as an unbiased test of that choice.
- Cloud Score+ composites and midpoint-dated half-month observations differ from the training inputs; detection here is coarser than in training.
- Pixels without a qualified, cloud-free window get no crop prediction, so coverage is partial by design.
- The model only knows carrot, cabbage and potato. Other crops, forest and built-up land can still receive confident crop labels.
- Local accuracy needs dated field observations collected under a separate sampling protocol.

In [ ]:
# Print the bundle identity needed to cite or reproduce this exploratory map
print("Status:", manifest["status"])
print("Model package:", manifest["model_manifest"])
print("Model SHA-256:", manifest["identity"]["model_sha256"])
print("Year:", manifest["identity"]["year"], "| Study pixels:", manifest["study_pixels"],
      "| Pixel area (ha):", manifest["pixel_ha"])
print("Main Paoay outputs, the model and the training partitions are unchanged.")